In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights


weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 1) 


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")
model = model.to(device)


class ImageRatingDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples 
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, rating = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        rating = torch.tensor([float(rating)], dtype=torch.float32)
        return image, rating

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.meta["mean"], std=weights.meta["std"]),
])

samples = [("/path/to/img1.jpg", 78), ("/path/to/img2.jpg", 42),]

dataset = ImageRatingDataset(samples, transform=transform)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.train()
for epoch in range(5):
    total_loss = 0.0
    for images, ratings in loader:
        images = images.to(device)
        ratings = ratings.to(device)

        preds = model(images)
        loss = criterion(preds, ratings)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1}: loss={total_loss/len(loader):.4f}")


model.eval()
with torch.no_grad():
    img = Image.open("/path/to/new_image.jpg").convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    pred = model(x).item()
    pred = max(0, min(100, pred)) 
    print("Predicted rating:", pred)

Using Device: cpu


KeyError: 'mean'